In [2]:
"Write program to implement decision tree"

'Write program to implement decision tree'

In [1]:
import numpy as np

class DecisionTreeClassifier:
    """A basic decision tree classifier using Gini impurity."""

    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree_ = None

    @staticmethod
    def _gini(y):
        _, counts = np.unique(y, return_counts=True)
        probabilities = counts / len(y)
        return 1.0 - np.sum(probabilities**2)

    @staticmethod
    def _majority_class(y):
        labels, counts = np.unique(y, return_counts=True)
        return labels[np.argmax(counts)]

    def _build_tree(self, X, y, depth):
        node = {"prediction": self._majority_class(y)}

        if (
            len(np.unique(y)) == 1
            or len(y) < self.min_samples_split
            or (self.max_depth is not None and depth >= self.max_depth)
        ):
            return node

        best_gain = 0.0
        best_split = None
        parent_impurity = self._gini(y)

        for feature in range(X.shape[1]):
            values = np.unique(X[:, feature])
            thresholds = (values[:-1] + values[1:]) / 2

            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = ~left_mask

                if not left_mask.any() or not right_mask.any():
                    continue

                left_y, right_y = y[left_mask], y[right_mask]
                weighted_impurity = (
                    len(left_y) * self._gini(left_y)
                    + len(right_y) * self._gini(right_y)
                ) / len(y)
                gain = parent_impurity - weighted_impurity

                if gain > best_gain:
                    best_gain = gain
                    best_split = (feature, threshold, left_mask, right_mask)

        if best_split is None:
            return node

        feature, threshold, left_mask, right_mask = best_split
        node.update({
            "feature": feature,
            "threshold": threshold,
            "left": self._build_tree(X[left_mask], y[left_mask], depth + 1),
            "right": self._build_tree(X[right_mask], y[right_mask], depth + 1),
        })
        return node

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        if X.ndim != 2 or y.ndim != 1 or len(X) != len(y):
            raise ValueError("X must be 2D and y must be 1D with matching rows.")
        if len(y) == 0:
            raise ValueError("Training data cannot be empty.")

        self.tree_ = self._build_tree(X, y, depth=0)
        return self

    def _predict_one(self, row, node):
        while "feature" in node:
            if row[node["feature"]] <= node["threshold"]:
                node = node["left"]
            else:
                node = node["right"]
        return node["prediction"]

    def predict(self, X):
        if self.tree_ is None:
            raise RuntimeError("Call fit before predict.")

        X = np.asarray(X, dtype=float)
        if X.ndim != 2:
            raise ValueError("X must be a 2D array.")

        return np.array([self._predict_one(row, self.tree_) for row in X])


# Example
X = [[1], [2], [3], [4], [5], [6]]
y = ["low", "low", "low", "high", "high", "high"]

model = DecisionTreeClassifier(max_depth=3).fit(X, y)
print(model.predict([[2.5], [5.5]]))

['low' 'high']
